In [8]:
import glob
import pickle
import numpy as np
import pandas as pd

OUT_DIR = "./output1.3"

result_paths = sorted(glob.glob(f"{OUT_DIR}/sim1_final_result*.pkl"))
if not result_paths:
    raise FileNotFoundError(f"No sim1_final result files found in {OUT_DIR}")

data = []
for path in result_paths:
    with open(path, "rb") as f:
        chunk = pickle.load(f)
    data.extend(chunk if isinstance(chunk, list) else [chunk])

thetas     = [7.8, 8.0, 8.2, 8.4, 8.6]
theta_keys = [f"{t:g}" for t in thetas]

# AIS config used in simulation_scenario1_final.py
AIS_N_PATHS         = 300
AIS_N_LEVELS        = 20
AIS_MOVES_PER_LEVEL = 5

print(f"Loaded {len(data)} replications, thetas={thetas}")


Loaded 50 replications, thetas=[7.8, 8.0, 8.2, 8.4, 8.6]


In [ ]:
data[0]["thetas"]


In [ ]:
def l1(a, b):
    return float(np.sum(np.abs(np.asarray(a) - np.asarray(b))))

def ci_iou(lo_m, hi_m, lo_ref, hi_ref):
    intersection = np.maximum(0, np.minimum(hi_m, hi_ref) - np.maximum(lo_m, lo_ref))
    ref_len = hi_ref - lo_ref
    return float(np.nanmean(np.where(ref_len > 0, intersection / ref_len, np.nan)))

def ess(normw):
    w = np.asarray(normw, dtype=float)
    return float(1.0 / np.sum(w ** 2))

def normw_from_logw(logw):
    lw = np.asarray(logw, float)
    w  = np.exp(lw - np.max(lw))
    return w / w.sum()

rows = []
for x in data:
    mcmc_m  = np.asarray(x["MCMC"])
    mcmc_lo = x["MCMC_quantiles"]["0.025"]
    mcmc_hi = x["MCMC_quantiles"]["0.975"]

    for key in theta_keys:
        # --- RPS ---
        rps_nw = normw_from_logw(x[f"RPS_{key}_logw"])
        rows.append(dict(
            method="RPS",
            theta=float(key),
            n_states=x[f"RPS_{key}_n_states"],
            ESS=ess(rps_nw),
            ESS_ratio=ess(rps_nw) / x[f"RPS_{key}_n_states"],
            L1_mean=l1(x[f"RPS_{key}"], mcmc_m),
            L1_q025=l1(x[f"RPS_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"RPS_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"RPS_{key}_quantiles"]["0.025"],
                       x[f"RPS_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"RPS_{key}_time", np.nan),
        ))

        # --- AIS ---
        ais_out = x[f"AIS_{key}_output"]
        e_ais   = ess(ais_out.normw)
        rows.append(dict(
            method="AIS",
            theta=float(key),
            n_states=len(ais_out.terminals),
            ESS=e_ais,
            ESS_ratio=e_ais / len(ais_out.terminals),
            L1_mean=l1(x[f"AIS_{key}"], mcmc_m),
            L1_q025=l1(x[f"AIS_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"AIS_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"AIS_{key}_quantiles"]["0.025"],
                       x[f"AIS_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"AIS_{key}_time", np.nan),
        ))

        # --- PB ---
        pb_nw = normw_from_logw(x[f"PB_{key}_logw"])
        rows.append(dict(
            method="PB",
            theta=float(key),
            n_states=x[f"PB_{key}_n_states"],
            ESS=ess(pb_nw),
            ESS_ratio=ess(pb_nw) / x[f"PB_{key}_n_states"],
            L1_mean=l1(x[f"PB_{key}"], mcmc_m),
            L1_q025=l1(x[f"PB_{key}_quantiles"]["0.025"], mcmc_lo),
            L1_q975=l1(x[f"PB_{key}_quantiles"]["0.975"], mcmc_hi),
            IoU=ci_iou(x[f"PB_{key}_quantiles"]["0.025"],
                       x[f"PB_{key}_quantiles"]["0.975"], mcmc_lo, mcmc_hi),
            runtime_s=x.get(f"PB_{key}_time", "<1"),
        ))
rows[0]
df = pd.DataFrame(rows)

summary = (
    df.groupby("method")
    [["n_states", "L1_mean", "L1_q025", "L1_q975", "IoU", "ESS", "ESS_ratio", "runtime_s"]]
    .mean()
    .round(4)
)
summary


TypeError: agg function failed [how->mean,dtype->object]

In [10]:
# Same table broken out by theta
summary_theta = (
    df.groupby(["theta", "method"])
    [["n_states", "L1_mean", "L1_q025", "L1_q975", "IoU", "ESS", "ESS_ratio", "runtime_s"]]
    .mean()
    .round(4)
)
summary_theta


n_states  L1_mean  L1_q025  L1_q975     IoU       ESS  \
theta method                                                          
7.8   AIS       300.00   0.2907   0.0194   0.0208  0.9979  220.2505   
      PB         33.70   1.0313   0.0869   0.0996  0.9810   33.7000   
      RPS        13.70   1.9816   2.0944   0.9492  0.8587   13.7000   
8.0   AIS       300.00   0.2534   0.0184   0.0198  0.9982  215.0535   
      PB         37.06   1.0644   0.0837   0.0970  0.9837   37.0600   
      RPS        17.06   2.1183   2.0776   0.9381  0.8632   17.0600   
8.2   AIS       300.00   0.2896   0.0188   0.0209  0.9980  218.8687   
      PB         41.00   1.0399   0.0726   0.0758  0.9874   41.0000   
      RPS        21.00   2.0714   0.1816   0.4706  0.9607   21.0000   
8.4   AIS       300.00   0.2614   0.0183   0.0205  0.9979  209.4565   
      PB         44.38   0.8578   0.0616   0.0693  0.9884   44.3800   
      RPS        24.38   1.8035   0.1862   0.4700  0.9624   24.3800   
8.6   AIS       300.00   0.2759   0.0195   0.0209  0.9981  207.2071   
      PB         47.64   0.7560   0.0535   0.0588  0.9904   47.6400   
      RPS        27.64   1.7003   0.1723   0.2909  0.9719   27.6400   

              ESS_ratio  runtime_s  
theta method                        
7.8   AIS        0.7342    42.6973  
      PB         1.0000     0.7068  
      RPS        1.0000        NaN  
8.0   AIS        0.7168    42.6831  
      PB         1.0000     0.7083  
      RPS        1.0000        NaN  
8.2   AIS        0.7296    42.7058  
      PB         1.0000     0.6683  
      RPS        1.0000        NaN  
8.4   AIS        0.6982    42.6834  
      PB         1.0000     0.6742  
      RPS        1.0000        NaN  
8.6   AIS        0.6907    42.6579  
      PB         1.0000     0.6443  
      RPS        1.0000        NaN